In [1]:
# imports
import time
import pandas as pd
import requests
from datetime import datetime
from riotwatcher import LolWatcher, ApiError
from pathlib import Path

# Walk up from cwd until we find the repo root (identified by .git folder).
# This works regardless of where Jupyter was launched from.
def _find_repo_root():
    p = Path().resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / ".git").exists():
            return candidate
    return p  # fallback: cwd

ROOT = _find_repo_root()
print(f"ROOT: {ROOT}")


ROOT: C:\Users\Laser\cs163-group18-winfactors


In [ ]:
import time
import requests
import pandas as pd
from collections import deque
from pathlib import Path
from riotwatcher import LolWatcher, ApiError

def _find_repo_root():
    p = Path().resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / ".git").exists():
            return candidate
    return p

ROOT    = _find_repo_root()
OUT_DIR = ROOT / "data" / "match_data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"OUT_DIR: {OUT_DIR}")

def match_routing(mid):
    if mid.startswith("NA1_"):  return "americas"
    if mid.startswith("EUW1_"): return "europe"
    if mid.startswith("KR_"):   return "asia"
    if mid.startswith("EUN1_"): return "europe"
    return "americas"

def load_key():
    for p in [
        Path.cwd() / "riot_api_key.txt",
        Path.cwd() / "src" / "riot_api_key.txt",
        ROOT / "riot_api_key.txt",
        ROOT / "src" / "riot_api_key.txt",
    ]:
        if p.exists():
            return p.read_text().strip()
    raise FileNotFoundError("riot_api_key.txt not found.")

api_key = load_key()
watcher = LolWatcher(api_key)
print(f"Loaded API key: {api_key[:10]}... (length: {len(api_key)})")

class KeyExpiredError(Exception):
    pass

def parse_match(match_json):
    meta = match_json.get("metadata", {})
    info = match_json.get("info", {})
    match_id = meta.get("matchId") or match_json.get("gameId")
    team_map = {}
    for t in info.get("teams", []):
        tid = t.get("teamId")
        obj = t.get("objectives", {})
        team_map[tid] = {
            "team_win":               t.get("win", False),
            "team_baron_kills":       obj.get("baron",      {}).get("kills", 0),
            "team_dragon_kills":      obj.get("dragon",     {}).get("kills", 0),
            "team_tower_kills":       obj.get("tower",      {}).get("kills", 0),
            "team_inhibitor_kills":   obj.get("inhibitor",  {}).get("kills", 0),
            "team_rift_herald_kills": obj.get("riftHerald", {}).get("kills", 0),
        }
    rows = []
    for p in info.get("participants", []):
        tid = p.get("teamId")
        team = team_map.get(tid, {})
        kills, deaths, assists = p.get("kills", 0), p.get("deaths", 0), p.get("assists", 0)
        rows.append({
            "match_id":                  match_id,
            "puuid":                     p.get("puuid"),
            "summonerName":              p.get("riotIdGameName") or p.get("summonerName"),
            "championName":              p.get("championName"),
            "championId":                p.get("championId"),
            "teamId":                    tid,
            "teamPosition":              p.get("teamPosition"),
            "role":                      p.get("role"),
            "win":                       p.get("win"),
            "kills":                     kills,
            "deaths":                    deaths,
            "assists":                   assists,
            "kda":                       (kills + assists) / max(1, deaths),
            "totalDamageToChampions":    p.get("totalDamageDealtToChampions", 0),
            "physicalDamageToChampions": p.get("physicalDamageDealtToChampions", 0),
            "magicDamageToChampions":    p.get("magicDamageDealtToChampions", 0),
            "totalDamageTaken":          p.get("totalDamageTaken", 0),
            "damageSelfMitigated":       p.get("damageSelfMitigated", 0),
            "damageToBuildings":         p.get("damageDealtToBuildings", 0),
            "damageToObjectives":        p.get("damageDealtToObjectives", 0),
            "goldEarned":                p.get("goldEarned", 0),
            "goldSpent":                 p.get("goldSpent", 0),
            "totalMinionsKilled":        p.get("totalMinionsKilled", 0),
            "neutralMinionsKilled":      p.get("neutralMinionsKilled", 0),
            "cs":                        p.get("totalMinionsKilled", 0) + p.get("neutralMinionsKilled", 0),
            "visionScore":               p.get("visionScore", 0),
            "wardsPlaced":               p.get("wardsPlaced", 0),
            "wardsKilled":               p.get("wardsKilled", 0),
            "visionWardsBought":         p.get("visionWardsBoughtInGame", 0),
            "timeCCingOthers":           p.get("timeCCingOthers", 0),
            "totalTimeCCDealt":          p.get("totalTimeCCDealt", 0),
            "champLevel":                p.get("champLevel"),
            "doubleKills":               p.get("doubleKills", 0),
            "tripleKills":               p.get("tripleKills", 0),
            "quadraKills":               p.get("quadraKills", 0),
            "pentaKills":                p.get("pentaKills", 0),
            "turretKills":               p.get("turretKills", 0),
            "firstBloodKill":            p.get("firstBloodKill", False),
            "firstBloodAssist":          p.get("firstBloodAssist", False),
            "longestTimeSpentLiving":    p.get("longestTimeSpentLiving"),
            "totalTimeSpentDead":        p.get("totalTimeSpentDead"),
            "spell1Casts":               p.get("spell1Casts", 0),
            "spell2Casts":               p.get("spell2Casts", 0),
            "spell3Casts":               p.get("spell3Casts", 0),
            "spell4Casts":               p.get("spell4Casts", 0),
            "team_win":                  team.get("team_win"),
            "team_baron_kills":          team.get("team_baron_kills", 0),
            "team_dragon_kills":         team.get("team_dragon_kills", 0),
            "team_tower_kills":          team.get("team_tower_kills", 0),
            "team_inhibitor_kills":      team.get("team_inhibitor_kills", 0),
            "team_rift_herald_kills":    team.get("team_rift_herald_kills", 0),
        })
    return rows

def fetch_match(region, mid):
    try:
        return watcher.match.by_id(region, mid)
    except ApiError as e:
        if e.response.status_code == 401: raise KeyExpiredError()
        print(f"  API error {e.response.status_code} for {mid}, skipping.")
        return None
    except KeyExpiredError:
        raise
    except Exception as exc:
        print(f"  Error fetching {mid}: {exc}")
        return None

def get_match_ids(puuid, region):
    ids, start = [], 0
    while True:
        try:
            batch = watcher.match.matchlist_by_puuid(region, puuid, start=start, count=100)
        except ApiError as e:
            if e.response.status_code == 401: raise KeyExpiredError()
            print(f"  [get_match_ids] API {e.response.status_code} for puuid={puuid[:12] if puuid else None} region={region}")
            break
        except Exception as exc:
            print(f"  [get_match_ids] error for region={region}: {exc}")
            break
        if not batch:
            break
        ids.extend(batch)
        start += 100
        time.sleep(0.4)
    return ids

def fetch_tier_entries(platform, tier, api_key):
    endpoints = {
        "challenger":  f"https://{platform}.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5",
        "grandmaster": f"https://{platform}.api.riotgames.com/lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5",
        "master":      f"https://{platform}.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5",
    }
    r = requests.get(endpoints[tier.lower()], headers={"X-Riot-Token": api_key})
    r.raise_for_status()
    entries = r.json().get("entries", [])
    print(f"    {tier.capitalize()}: {len(entries)} players")
    return entries

def collect(
    platforms=(("na1", "americas"), ("euw1", "europe"), ("kr", "asia")),
    tiers=("challenger", "grandmaster", "master"),
    max_players_per_region=None,
    out_file="challenger_matches.csv",
    save_every=1000,
    print_every=10,
    matches_per_turn=50,          # swap region after this many matches
    matches_per_player_visit=10,  # matches taken per player visit before re-queuing
):
    out_path = OUT_DIR / out_file

    if out_path.exists():
        existing = pd.read_csv(out_path)
        all_rows = existing.to_dict("records")
        seen = set(existing["match_id"].dropna().unique())
        print(f"Resuming: {len(seen)} matches already collected ({len(all_rows)} rows)")
    else:
        all_rows, seen = [], set()
        print(f"Starting fresh → {out_path}")

    # ── Build per-region player queues ─────────────────────────────────────────
    player_queues = {}
    for platform, region in platforms:
        print(f"\n{platform.upper()} ({region}):")
        entries, seen_puuids = [], set()
        for tier in tiers:
            try:
                for e in fetch_tier_entries(platform, tier, api_key):
                    uid = e.get("puuid") or e.get("summonerId") or id(e)
                    if uid not in seen_puuids:
                        entries.append({"puuid": e.get("puuid"),
                                        "summonerId": e.get("summonerId"),
                                        "platform": platform,
                                        "region": region,
                                        "_tier": tier,
                                        "pending_ids": None})
                        seen_puuids.add(uid)
            except Exception as exc:
                print(f"    Failed to fetch {tier}: {exc}")

        if max_players_per_region:
            entries = entries[:max_players_per_region]
        player_queues[region] = deque(entries)
        print(f"  → {len(entries)} unique players queued")

    region_order = [r for _, r in platforms]
    match_queues = {r: deque() for r in region_order}

    # ── Startup summary so you can see if any region failed to build ───────────
    print(f"\n{'='*50}")
    print(f"Player queues: " + ", ".join(f"{r}={len(player_queues[r])}" for r in region_order))
    print(f"{'='*50}\n")

    def refill(region):
        """Pop front player, take up to matches_per_player_visit new IDs,
        re-append to back of queue if they have more matches remaining."""
        while player_queues[region]:
            entry    = player_queues[region].popleft()
            platform = entry["platform"]
            puuid    = entry.get("puuid")

            if not puuid:
                try:
                    summ  = watcher.summoner.by_id(platform, entry["summonerId"])
                    puuid = summ["puuid"]
                    entry["puuid"] = puuid
                    time.sleep(0.4)
                except KeyExpiredError:
                    raise
                except Exception as exc:
                    print(f"  Couldn't resolve puuid for {region}: {exc}")
                    continue

            if entry["pending_ids"] is None:
                print(f"  [{entry['_tier']}@{platform}] {puuid[:12]}... fetching IDs", end="", flush=True)
                all_ids = get_match_ids(puuid, region)
                entry["pending_ids"] = [m for m in all_ids if m not in seen]
                print(f" → {len(all_ids)} total, {len(entry['pending_ids'])} new")

            if not entry["pending_ids"]:
                continue  # exhausted, try next player

            batch = entry["pending_ids"][:matches_per_player_visit]
            entry["pending_ids"] = entry["pending_ids"][matches_per_player_visit:]
            match_queues[region].extend(batch)

            if entry["pending_ids"]:
                player_queues[region].append(entry)
            return  # one player visit per call

    def save():
        pd.DataFrame(all_rows).to_csv(out_path, index=False)

    # ── Main loop — round-robin at match level ─────────────────────────────────
    region_idx = 0
    turn_counts = {r: 0 for r in region_order}

    while any(match_queues[r] or player_queues[r] for r in region_order):
        region = None
        for _ in range(len(region_order)):
            r = region_order[region_idx % len(region_order)]
            region_idx += 1
            if match_queues[r] or player_queues[r]:
                region = r
                break
        if region is None:
            break

        try:
            while len(match_queues[region]) < matches_per_turn and player_queues[region]:
                refill(region)
        except KeyExpiredError:
            save()
            print(f"\nKey expired. Saved {len(seen)} matches. Update riot_api_key.txt and re-run.")
            return None

        if not match_queues[region]:
            print(f"  !! {region}: no new matches found — all players exhausted or API errors. Skipping.")
            continue

        turn_counts[region] += 1
        print(f"\n{'─'*40}")
        print(f"  {region.upper()} turn #{turn_counts[region]}  ({min(matches_per_turn, len(match_queues[region]))} matches)  "
              f"[queues: " + ", ".join(f"{r}={len(match_queues[r])}p+{len(player_queues[r])}pl" for r in region_order) + "]")
        print(f"{'─'*40}")

        count = 0
        while match_queues[region] and count < matches_per_turn:
            mid = match_queues[region].popleft()
            try:
                data = fetch_match(match_routing(mid), mid)
            except KeyExpiredError:
                save()
                print(f"\nKey expired. Saved {len(seen)} matches. Update riot_api_key.txt and re-run.")
                return None
            if data:
                try:
                    rows = parse_match(data)
                    all_rows.extend(rows)
                    seen.add(mid)
                except Exception as exc:
                    print(f"  Parse error {mid}: {exc}")
            count += 1
            time.sleep(0.4)

            total = len(seen)
            if total % print_every == 0:
                print(f"  [{total} matches] {region} {count}/{matches_per_turn}: {mid}")
            if total % save_every == 0:
                save()
                print(f"  ── saved {total} matches ──")

    save()
    df = pd.DataFrame(all_rows)
    print(f"\nDone. {len(seen)} matches, {len(df)} rows → {out_path}")
    print("Turn counts: " + ", ".join(f"{r}={n}" for r, n in turn_counts.items()))
    return df

# ── Run ────────────────────────────────────────────────────────────────────────
df = collect(
    platforms=(("na1", "americas"), ("euw1", "europe"), ("kr", "asia")),
    tiers=("challenger", "grandmaster", "master"),
    out_file="challenger_matches.csv",
)


OUT_DIR: C:\Users\Laser\cs163-group18-winfactors\data\match_data
Loaded API key: RGAPI-86a8... (length: 42)


C:\Users\Laser\AppData\Local\Temp\ipykernel_15468\3439400578.py:178: DtypeWarning: Columns (45) have mixed types. Specify dtype option on import or set low_memory=False.
  existing = pd.read_csv(out_path)


Resuming: 39592 matches already collected (409771 rows)

NA1 (americas):
    Failed to fetch challenger: 401 Client Error: Unauthorized for url: https://na1.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5
    Failed to fetch grandmaster: 401 Client Error: Unauthorized for url: https://na1.api.riotgames.com/lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5
    Failed to fetch master: 401 Client Error: Unauthorized for url: https://na1.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5
  → 0 unique players queued

EUW1 (europe):
    Challenger: 300 players
    Failed to fetch grandmaster: 401 Client Error: Unauthorized for url: https://euw1.api.riotgames.com/lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5
    Master: 10000 players
  → 10300 unique players queued

KR (asia):
    Challenger: 300 players
    Failed to fetch grandmaster: 401 Client Error: Unauthorized for url: https://kr.api.riotgames.com/lol/league/v4/grandmas

In [ ]:
# Test the API key with a simple request
# Run this cell to verify your Riot API key works before running the main pipeline.

import requests

try:
    api_key = load_key()
    print(f"Testing key: {api_key[:10]}... (length: {len(api_key)})")
    headers = {"X-Riot-Token": api_key}
    # Test with a known summoner (change 'Doublelift' to any valid name if needed)
    test_url = "https://na1.api.riotgames.com/lol/summoner/v4/summoners/by-name/Doublelift"
    r = requests.get(test_url, headers=headers)
    print(f"Response status: {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print(f"Success! Summoner: {data.get('name')}, Level: {data.get('summonerLevel')}")
    else:
        print(f"Failed: {r.text}")
except Exception as e:
    print(f"Error testing key: {e}")